# Generating Samples

In this notebook we will show how to generate valid samples for a given parser without using a grammar.

## Examples

First we import the convenience utilities.

In [ ]:
import src.utils as utils

### CGIencode.py

In [ ]:
cgiencode = utils.load_file('subjects/cgiencode.py', 'cgiencode')

In [ ]:
cgidecode = utils.load_file('subjects/cgidecode.py', 'cgidecode')

## The error handler

We often need to interpret the error we get back. We use a simple exception class to capture the error.

In [ ]:
with utils.ExpectError():
     s = cgiencode.main('ab cd')
     print(s)
     print(cgidecode.main(s))

In [ ]:
with utils.ExpectError():
     s = cgiencode.main('http://www.google.com/search?q=fuzzing')
     print(s)
     print(cgidecode.main(s))

In [ ]:
with utils.ExpectError():
     s = cgiencode.main('https://www.google.com/search?client=firefox-b-d&q=ASE+2026')
     print(s)
     print(cgidecode.main(s))

## A random fuzzer.

In [ ]:
import random
random.seed(0)

In [ ]:
import string

In [ ]:
def fuzzer(max_length=100):
    string_length = random.randrange(1, max_length + 1)
    return ''.join([random.choice(string.printable) for c in range(string_length)])

In [ ]:
fuzzer()

What happens if you feed this input to the program?

In [ ]:
with utils.ExpectError():
    s = fuzzer()
    print(repr(s))
    r = cgiencode.main(s)
    print(repr(r))

In [ ]:
with utils.ExpectError():
    for i in range(10):
        s = fuzzer()
        r = cgiencode.main(s)
        l, v = cgidecode.main(r)
        print(repr(s))
        print(repr(v))
        print(s == v)
        print()

__QUESTION:__ Why the difference? Can you identify the reason?

## Tracking Coverage

Usage
```
with Coverage() as cov:
   function_to_be_traced()
c = cov.coverage()
```

In [ ]:
import sys, inspect
class Coverage:
    def __init__(self):
        self._trace = []
        self._unseen = {}

    # Trace function
    # Tracing function. To be overloaded in subclasses.
    def traceit(self, frame, event, arg):
        if self.original_trace_function is not None:
            self.original_trace_function(frame, event, arg)

        if event == "line":
            function_name = frame.f_code.co_name
            lineno = frame.f_lineno
            if function_name != '__exit__':  # avoid tracing ourselves:
                self._trace.append((function_name, lineno))

        return self.traceit
        
    # Start of `with` block. Turn on tracing.
    def __enter__(self):
        self.original_trace_function = sys.gettrace()
        sys.settrace(self.traceit)
        return self

    # End of `with` block. Turn off tracing.
    def __exit__(self, exc_type, exc_value, tb):
        sys.settrace(self.original_trace_function)
        return None  # default: pass all exceptions

    # The list of executed lines, as (function_name, line_number) pairs
    def trace(self):
        return self._trace

    # The set of executed lines, as (function_name, line_number) pairs
    def coverage(self):
        return set(self.trace())

    # The set of function names seen
    def function_names(self):
        return set(function_name for (function_name, line_number) in self.coverage())

    # eturn a string representation of this object.
    # Show covered (and uncovered) program code
    def __repr__(self):
        t = ""
        self.unseen = []
        for function_name in self.function_names():
            if function_name in dir(__builtins__): continue
            # Similar code as in the example above
            try:
                fun = eval(function_name)
            except Exception as exc:
                t += f"Skipping {function_name}: {exc}"
                continue

            source_lines, start_line_number = inspect.getsourcelines(fun)
            for lineno in range(start_line_number, start_line_number + len(source_lines)):
                if (function_name, lineno) not in self.trace():
                    t += "# "
                    self._unseen[(function_name, lineno)] = False
                else:
                    t += "  "
                    self._unseen[(function_name, lineno)] = True
                t += "%2d  " % lineno
                t += source_lines[lineno - start_line_number]

        return t

In [ ]:
from subjects.cgiencode import cgi_encode

In [ ]:
print(repr(r))

In [ ]:
with Coverage() as cov:
    cgi_encode('ab?cd=p&q;')

In [ ]:
print(cov)

In [ ]:
print([c for c in cov._unseen if not cov._unseen[c]])

In [ ]:
s = fuzzer()
print(repr(s))
with cov as cov:
    r = cgiencode.cgi_encode(s)
print(repr(r))
res = repr(cov)
unseen = [c for c in cov._unseen if not cov._unseen[c]]
for u in unseen:
    print(u)

Let us consider a more complex program

### Calculator.py

In [ ]:
from subjects.calculator import parse_expr
cov = Coverage()

In [ ]:
s = fuzzer()
print(repr(s))
with utils.ExpectError():
    with cov as cov:
        r = parse_expr(s)
print(repr(r))
res = repr(cov)
unseen = [c for c in cov._unseen if not cov._unseen[c]]
for u in unseen:
    print(u)

This is rather unsatisfying. We need a better way to reach deeper into the program. Let us observe the error again, this time with a plausible partial input.

In [ ]:
calculator = utils.load_file('subjects/calculator.py', 'calculator')

In [ ]:
with utils.ExpectError():
     calculator.main('(1+2)de')

As you can see, the exception we got precisely indicates **exactly where the parse error occurred**.

In [ ]:
'(1+2)de'[0:5]

We will next see how to leverage this parse error feedback.

## Leveraging Feedback

Can we make use of the feedback from the fuzzer to construct better inputs? let us write a method to capture the exception details

In [ ]:
import enum

We distinguish three kinds of parse results.

In [ ]:
class ExprStatus(enum.Enum):
    Complete = 0
    Unterminated = -1
    Unexpected = -2

We extend our `ExpectError` to extract extra details

In [ ]:
class ExpectExprError(utils.ExpectError):
    def __init__(self, s, log=False):
        self.boundary = None
        self.result = None
        self.s = s
        self.log = log

    def __exit__(self, exc_type, exc_value, tb):
        if exc_type is None:
            self.boundary = 0
            self.result = ExprStatus.Complete
            return
        inp, self.boundary = exc_value.args
        if self.boundary >= len(self.s):
            self.result = ExprStatus.Unterminated
        elif self.boundary < len(self.s):
            self.result =  ExprStatus.Unexpected
        else:
            assert False
        return True

Let us see how it works.

In [ ]:
with ExpectExprError('(1+2x)') as e:
     calculator.main(e.s)
e.boundary, e.result

That is, at position `4` we found an unexpected character.

In [ ]:
with ExpectExprError('(1+2') as e:
     calculator.main(e.s)
e.boundary, e.result

This indicates, at position `4` we were still expecting data, but none was found.

In [ ]:
with ExpectExprError('(1+2)') as e:
     calculator.main(e.s)
e.boundary, e.result

We have a valid input.

In [ ]:
with ExpectExprError('(1+2)x') as e:
     calculator.main(e.s)
e.boundary, e.result

That is, at the index `5` we found an unexpected character.

## Feedback Based Sample Generator

We can leverage this information to generate inputs quickly. The idea is that:
1. If the input is unterminated, we can extend it further
2. If the input is unexpected, we find the parse boundary, remove the extra details, and extend it with a new suffix
3. If the input is valid, we are done

In [ ]:
def validate_calculator(input_str, log_level):
    with ExpectExprError(input_str) as e:
        calculator.main(e.s)
    match e.result:
        case ExprStatus.Complete:
            return ExprStatus.Complete, len(input_str), None
        case ExprStatus.Unexpected:
            return ExprStatus.Unexpected, e.boundary, None
        case ExprStatus.Unterminated:
            return ExprStatus.Unterminated, e.boundary, None
    return None

Let us test it

In [ ]:
s = 'H'
with ExpectExprError(s) as e:
    calculator.main(e.s)
print(e.result, e.boundary)

In [ ]:
validate_calculator(s, 0)

In [ ]:
s = '(1+1'
with ExpectExprError(s) as e:
    calculator.main(e.s)
print(e.result, e.boundary)

In [ ]:
validate_calculator(s, 0)

In [ ]:
s = '(1+1x'
with ExpectExprError(s) as e:
    calculator.main(e.s)
print(e.result, e.boundary)

In [ ]:
validate_calculator(s, 0)

Let us now write a simple genertor

In [ ]:
import string
import random

In [ ]:
def get_next_char(log_level):
    # set_of_chars = ['[',']','{','}','(',')','<','>','1','0','a','b',':','"',',','.', '\'']
    set_of_chars = string.printable
    idx = random.randrange (0,len(set_of_chars),1)
    input_char = set_of_chars[idx]
    if (log_level):
        print(input_char)
    return input_char

In [ ]:
def generate(log_level):
    """
    Feed it one character at a time, and see if the parser rejects it. 
    If it does not, then append one more character and continue. 
    If it rejects, replace with another character in the set. 
    :returns completed string
    """
    prev_str = ""
    while True:
        char = get_next_char(log_level)
        curr_str = prev_str + str(char)
        rv, n, c = validate_calculator(curr_str, log_level)
        if log_level:
            print("LOG: %s n=%d, c=%s. Input string is <<%s>>" % (rv,n,c,curr_str))
        if rv == ExprStatus.Complete:
            return curr_str
        elif rv == ExprStatus.Unterminated: # go ahead...
            prev_str = curr_str
            continue
        elif rv == ExprStatus.Unexpected: # try again with a new random character do not save current character
            continue
        else:
            print("ERROR!")
            break
    return None

Let us try it out

In [ ]:
generate(0)

In [ ]:
def create_valid_strings(n, log_level):
    i = 0
    while True:
        created_string = generate(log_level)
        if created_string is not None:
            print(repr(created_string))
            i = i + 1
            if (i >= n):
                break

In [ ]:
create_valid_strings(10, 0)

## Building the Evolutionary Algorithm

Can we do better than that? Here is an idea to leverage evolutionary algorithms. Bug first, we need to come up with some scoring rules. The idea is to
try and minimize the score.

1. If the input is valid, then we need a very small score. We set it to a fraction 1/length of the input. This way, we encourage the larger inputs.
2. If the input is unterminated, this is still reasonable. We want it to be scored `1`
3. If the input is unexpected, then we want to have a large score so that the system needs to actively minimize it.

In [ ]:
def get_expr_fitness(s):
    with ExpectExprError(s) as e:
        calculator.main(e.s)
    match e.result:
        case ExprStatus.Complete:
            return 1.0/len(e.s)
        case ExprStatus.Unexpected:
            return len(e.s) - e.boundary
        case ExprStatus.Unterminated:
            return 1
    assert False, (s, e)

In [ ]:
get_expr_fitness('(1+2)')

In [ ]:
get_expr_fitness('(1+(2*4+4))')

In [ ]:
get_expr_fitness('(1+2)234')

In [ ]:
get_expr_fitness('(1+2+3')

In [ ]:
get_expr_fitness('(1+2+3XXY')

### The Evolver class

In [ ]:
class Evolver:
    def __init__(self, delta=0.1, log=True):
        self.log = log
        self.delta = delta
        self.tournament_size = 10
        self.crossover_chance = 0.7

Creating an initial population

In [ ]:
class Evolver(Evolver):
    def create_population(self, size):
        return [fuzzer() for i in range(size)]

In [ ]:
expr_evolver = Evolver()
expr_evolver.get_fitness = get_expr_fitness

In [ ]:
expr_evolver.create_population(10)

#### Mutation

We let the probability of mutaion to be 1/length of the string.

In [ ]:
class Evolver(Evolver):
    def mutate(self, chromosome):
        P = 1.0 / len(chromosome)
        new_chromosome = [c if random.random() >= P
                          else
                          chr(int(random.gauss(ord(chromosome[pos]), 100) % 65536))
                for pos,c in enumerate(chromosome)]
        return ''.join(new_chromosome)

In [ ]:
expr_evolver = Evolver()
expr_evolver.get_fitness = get_expr_fitness
for i in range(10):
    print(expr_evolver.mutate('11111111'))

### Tournament

We now select the winner from a small cohort.

We cross two selected individuals to get their offsprint.

In [ ]:
class Evolver(Evolver):
    def crossover(self, parent1, parent2):
        pos = random.randint(1, len(parent1))
        offspring1 = parent1[:pos] + parent2[pos:]
        offspring2 = parent2[:pos] + parent1[pos:]
        return [offspring1, offspring2]

In [ ]:
expr_evolver = Evolver()
expr_evolver.get_fitness = get_expr_fitness
for i in range(10):
    print(expr_evolver.crossover('1111111111', '2222222222'))

In [ ]:
class Evolver(Evolver):
    def selection(self, competition):
        return min(competition, key=lambda individual: individual[1])[0]

In [ ]:
class Evolver(Evolver):
    def regenerate_population(self, fitness):
        new_population = []
        while len(new_population) < len(fitness):
            # Selection -- best of self.tournament_size
            offsprings = [self.selection(random.sample(fitness, self.tournament_size))
                          for i in range(2)]
            # Crossover
            if random.random() < self.crossover_chance:
                offsprings = self.crossover(*offsprings)

            # Mutation
            offsprings = [self.mutate(i) for i in offsprings]

            new_population.extend(offsprings)
        return new_population

In [ ]:
class Evolver(Evolver):
    def genetic_algorithm(self):
        generation = 0
        population = self.create_population(100)
        while True:
            fitness = []
            for x in population:
                    fitness.append((x, self.get_fitness(x)))
            best_individual, best_fitness = min(fitness, key=lambda item: item[1])
            if self.log:
                print("Best fitness of population: %s - %.10f" %
                      (repr(best_individual), best_fitness))

            # Stop when optimum found, or we run out of patience
            if best_fitness <= self.delta: break
            if generation > 1000: break
            # The next generation will have the same size as the current one
            population = self.regenerate_population(fitness)
            generation += 1

        if self.log:
            print("Best individual: %s, fitness %.10f" %(repr(best_individual), best_fitness))
        return best_individual, best_fitness

In [ ]:
expr_evolver = Evolver()
expr_evolver.get_fitness = get_expr_fitness

In [ ]:
expr_evolver.genetic_algorithm()

In [ ]:
class ExprEvolver(Evolver):
    def get_fitness(self, s):
        with ExpectExprError(s, log=self.log) as e:
            calculator.main(e.s)
        match e.result:
            case ExprStatus.Complete:
                return 1.0/len(e.s)
            case ExprStatus.Unexpected:
                return len(e.s) - e.boundary
            case ExprStatus.Unterminated:
                return 1
        assert False, (s, e)

In [ ]:
expr_evolver = ExprEvolver(log=False)

In [ ]:
for i in range(10):
    v = expr_evolver.genetic_algorithm()
    print(repr(v))

### JSON
Generating JSON can be slow. (Only if we have enough time).

In [ ]:
class JStatus(enum.Enum):
    Complete = 0
    Extra = 1
    Unterminated = -1
    Expecting = -2

In [ ]:
import subjects.microjson as microjson

In [ ]:
class ExpectJSONError:
    def __init__(self, s=None, log=False):
        self.msg = None
        self.boundary = None
        self.result = None
        self.s = s
        self.log = log

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, tb):
        if exc_type is None:
            self.boundary = 0
            self.result = JStatus.Complete
            return
        json_error = exc_value
        msg = str(exc_value)
        if self.log:
            print(msg, file=sys.stderr)
        if msg.startswith('extra data after JSON at position'):
            self.result = JStatus.Extra
        elif msg.startswith('malformed JSON data at position'):
            self.result = JStatus.Expecting
        elif msg.startswith('missing colon after key at position'):
            self.result = JStatus.Expecting
        elif msg.startswith('expected null at position'):
            self.result = JStatus.Expecting
        elif msg.startswith('expected boolean at position'):
            self.result = JStatus.Expecting
        elif msg.startswith('truncated JSON data at position'):                                                       
            self.result = JStatus.Unterminated
        else:
            # Not all exceptions have been specifically caught in the interest of simplicity.
            # assert False, msg
            self.result = JStatus.Expecting
        self.boundary = exc_value.pos
        return True

In [ ]:
error_data = [
    #expected null at position 0, "'n%m\ri<Q8P<t{STo~V&iH|_pJu}8_*fB\r'"
    'n%m\ri<Q8P<t{STo~V&iH|_pJu}8_*fB\r',
    # expected boolean at position 0, "'tWI6n )AB/'"
    'tWI6n )AB/',
    # missing colon after key at position 36, "'fn1+"AC8fwp{@cQ'"
    'fn1+"AC8fwp{@cQ'
]

In [ ]:
for x in error_data:
    with ExpectJSONError(x) as e:
        microjson.main(e.s)
    print(e.boundary, e.result)

In [ ]:
with ExpectJSONError() as e:
     microjson.main('["abc"]de')
e.boundary, e.result

In [ ]:
with ExpectJSONError() as e:
     microjson.main('["abc')
e.boundary, e.result

In [ ]:
with ExpectJSONError() as e:
     microjson.main('[ab')
e.boundary, e.result

In [ ]:
with ExpectJSONError() as e:
     microjson.main('[1,2,3]')
e.boundary, e.result

In [ ]:
class JSONEvolver(Evolver):
    def get_fitness(self, s):
        with ExpectJSONError(s, self.log) as e:
            microjson.main(e.s)
        match e.result:
            case JStatus.Complete:
                return 1.0/len(e.s)
            case JStatus.Extra:
                return len(s) - e.boundary
                # better to be incomplete than incorrect.
                return len(s) * 0.1
            case JStatus.Expecting:
                if len(s) == e.boundary:
                    return 1
                return len(s) - e.boundary
            case JStatus.Unterminated:
                return 1
        assert False, (s, e)

In [ ]:
json_evolver = JSONEvolver(log=False)

**Can be really slow**

In [ ]:
for i in range(10):
    v = json_evolver.genetic_algorithm()
    print(repr(v))

# Done

In [ ]:
#%tb